# Laboratorio Semana 4 · Selección y filtrado crítico con Pandas
**Ciencia de datos con Python — SENA (material de apoyo GFPI-F-135)**  
**Caso:** seguimiento académico simulado. Continuidad: Semana 2 (carga e inspección) → Semana 3 (calidad y trazabilidad) → Semana 4 (subconjuntos e interpretación).

**Regla de trabajo:** utilice exclusivamente datos ficticios; no reporte cifras como hechos institucionales. El archivo fue **preparado**, no certificado. Conserve los casos conflictivos para revisión y distinga *registros* de *personas únicas*.

## 0. Preparar ambiente y rutas relativas
Ubique el CSV entregado en `data/processed/seguimiento_academico_semana3_preparado.csv`. Ejecute todas las celdas de arriba hacia abajo. Requiere `pandas` y `ipykernel` si usa Jupyter.

In [1]:
from pathlib import Path
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FUENTE = RAIZ / "data" / "processed" / "seguimiento_academico_semana3_preparado.csv"
SALIDA = RAIZ / "outputs"
SALIDA.mkdir(parents=True, exist_ok=True)
assert FUENTE.exists(), f"No se encontró la fuente de la Semana 3: {FUENTE}"

df = pd.read_csv(
    FUENTE,
    dtype={"documento": "string", "ficha": "string"},
    parse_dates=["fecha_actualizacion"],
    encoding="utf-8-sig"
)
for campo in ["nota_python", "nota_datos", "asistencia"]:
    df[campo] = pd.to_numeric(df[campo], errors="coerce")
df["programa_normalizado"] = df["programa_normalizado"].astype("string")
df["ciudad_normalizada"] = df["ciudad_normalizada"].astype("string")
print("Pandas:", pd.__version__)
print("Filas y columnas:", df.shape)
print(df[["fila_fuente", "documento", "nombre", "programa_normalizado", "asistencia"]].head(5).to_string(index=False))

Pandas: 3.0.6
Filas y columnas: (19, 15)
 fila_fuente documento nombre programa_normalizado  asistencia
           2      1001    Ana                 ADSO        92.0
           3      1002 Carlos                 ADSO        85.0
           4      1003  María                 ADSO        96.0
           5      1004   Luis                 ADSO        70.0
           6      1005  Sofía                 ADSO        98.0


## 1. Recuperación de calidad: casos pendientes de la Semana 3
Una copia exacta fue retirada y quedó documentada. Sin embargo, **una clave de documento puede seguir repetida con calificaciones diferentes**. No decida cuál valor es verdadero: documente el conflicto. Los totales por filas no son automáticamente totales de personas.

In [2]:
repetido = df["documento"].notna() & df["documento"].duplicated(keep=False)
print("Total registros preparados:", len(df))
print("Registros con clave conflictiva:", int(repetido.sum()))
print(df.loc[repetido, ["fila_fuente", "documento", "nombre", "nota_python", "nota_datos"]].to_string(index=False))

# Exclusión SOLO del subconjunto de análisis que requiere identidad única:
base_unica = df.loc[df["documento"].notna() & ~repetido].copy()
print("Registros aptos según la regla didáctica de unicidad:", len(base_unica))

Total registros preparados: 19
Registros con clave conflictiva: 2
 fila_fuente documento nombre  nota_python  nota_datos
           8      1007 Camila          4.0         4.1
          21      1007 Camila          4.2         4.1
Registros aptos según la regla didáctica de unicidad: 17


## 2. Selección de columnas, loc e iloc
- `df["columna"]` devuelve Series; `df[["A", "B"]]` devuelve DataFrame.
- `.loc[filas, columnas]` selecciona etiquetas / condiciones.
- `.iloc[filas, columnas]` selecciona posiciones enteras; el extremo derecho del corte posicional es exclusivo. No suponga que índices y posiciones son iguales.

In [3]:
print("Serie:", type(df["nombre"]).__name__)
print("Tabla:", type(df[["documento", "nombre"]]).__name__)
print("loc con columnas:")
print(df.loc[:, ["documento", "nombre", "asistencia"]].head(4).to_string(index=False))
print("iloc con cuatro primeras filas y tres primeras columnas:")
print(df.iloc[:4, :3].to_string(index=False))

# Selección de filas y columnas por condición:
print(df.loc[df["documento"].eq("1004"), ["documento", "nombre", "asistencia"]].to_string(index=False))

Serie: Series
Tabla: DataFrame
loc con columnas:
documento nombre  asistencia
     1001    Ana        92.0
     1002 Carlos        85.0
     1003  María        96.0
     1004   Luis        70.0
iloc con cuatro primeras filas y tres primeras columnas:
 fila_fuente documento nombre
           2      1001    Ana
           3      1002 Carlos
           4      1003  María
           5      1004   Luis
documento nombre  asistencia
     1004   Luis        70.0


## 3. Filtros booleanos y reglas explícitas
Conserve siempre los registros con datos desconocidos por separado. Un valor ausente no equivale a asistencia de cero y no debe considerarse bajo asistencia automáticamente.

In [4]:
umbral_asistencia = 80  # Regla DIDÁCTICA, no umbral institucional oficial
mascara_baja = df["asistencia"].notna() & (df["asistencia"] < umbral_asistencia)
baja_asistencia = df.loc[mascara_baja, ["documento", "nombre", "programa_normalizado", "asistencia"]].copy()
print("Registros bajo umbral:", len(baja_asistencia))
print(baja_asistencia.to_string(index=False))

pendientes_asistencia = df.loc[df["asistencia"].isna(), ["documento", "nombre", "asistencia"]]
print("Registros sin asistencia verificable:", len(pendientes_asistencia))

Registros bajo umbral: 2
documento nombre programa_normalizado  asistencia
     1004   Luis                 ADSO        70.0
     1008   Juan                 ADSO        75.0
Registros sin asistencia verificable: 3


## 4. isin() y between()
`isin()` busca categorías dentro de una colección; `between()` usa límites **incluyentes** por defecto. Verifique explícitamente los valores ausentes antes de computar porcentajes.

In [5]:
programas = ["ADSO", "Ciencia de datos con Python"]
seleccion_programas = df.loc[df["programa_normalizado"].isin(programas),
    ["documento", "programa_normalizado", "asistencia"]]
print("Filas en programas definidos:", len(seleccion_programas))

rango_nota = df["nota_python"].notna() & df["nota_python"].between(3.0, 4.0, inclusive="both")
notas_intervalo = df.loc[rango_nota, ["documento", "nombre", "nota_python"]]
print("Registros con nota Python entre 3.0 y 4.0 (inclusive):", len(notas_intervalo))
print(notas_intervalo.to_string(index=False))

Filas en programas definidos: 19
Registros con nota Python entre 3.0 y 4.0 (inclusive): 7
documento nombre  nota_python
     1002 Carlos          3.8
     1006 Andrés          3.5
     1007 Camila          4.0
     1008   Juan          3.2
     1010  Diego          3.1
     1016  Mateo          3.6
     1017    NaN          4.0


## 5. Operadores lógicos `&`, `|` y `~`
Encierre **cada comparación** entre paréntesis. `&` es Y; `|` es O; `~` invierte una condición. Distinga los subconjuntos de registros en conflicto antes de emitir conclusiones.

In [6]:
# Regla 1: asistencia registrada menor de 80 Y programa ADSO
mascara_y = (
    df["programa_normalizado"].eq("ADSO")
    & df["asistencia"].notna()
    & df["asistencia"].lt(umbral_asistencia)
)
consulta_y = df.loc[mascara_y, ["documento", "nombre", "asistencia"]]
print("ADSO y asistencia menor de 80:", len(consulta_y))

# Regla 2: asistencia baja O nota Python conocida inferior a 3.5
mascara_o = (
    (df["asistencia"].notna() & df["asistencia"].lt(80))
    | (df["nota_python"].notna() & df["nota_python"].lt(3.5))
)
consulta_o = df.loc[mascara_o, ["documento", "nombre", "nota_python", "asistencia"]]
print("Coincide con al menos un criterio:", len(consulta_o))

# Regla 3: NO pertenecer al programa ADSO, solo entre programas informados
mascara_no = df["programa_normalizado"].notna() & ~(df["programa_normalizado"].eq("ADSO"))
consulta_no = df.loc[mascara_no, ["documento", "programa_normalizado"]]
print("Otros programas conocidos:", len(consulta_no))

ADSO y asistencia menor de 80: 2
Coincide con al menos un criterio: 3
Otros programas conocidos: 2


## 6. query() y variables externas
Use `@variable` para incorporar valores definidos en Python. **Nunca ejecute cadenas de consulta construidas directamente con texto no confiable que provenga de usuarios.**

In [7]:
programa_objetivo = "ADSO"
minima = 80
consulta_query = df.query("programa_normalizado == @programa_objetivo and asistencia < @minima", engine="python")
consulta_booleana = df.loc[
    df["programa_normalizado"].eq(programa_objetivo)
    & df["asistencia"].notna() & df["asistencia"].lt(minima)
]
assert set(consulta_query.index) == set(consulta_booleana.index)
print("query y máscara producen los mismos índices:", len(consulta_query))

query y máscara producen los mismos índices: 2


## 7. Validación del subconjunto y lectura crítica
Formule el universo, justifique exclusiones, verifique condiciones y documente recuentos. Si dos filas corresponden a un documento conflictivo, no afirme que representan dos aprendices.

In [8]:
base_sin_conflictos = df.loc[df["documento"].notna() & ~repetido].copy()
criterio = (
    base_sin_conflictos["programa_normalizado"].eq("ADSO")
    & base_sin_conflictos["asistencia"].notna()
    & base_sin_conflictos["asistencia"].lt(80)
)
subconjunto_validado = base_sin_conflictos.loc[criterio,
    ["fila_fuente", "documento", "nombre", "programa_normalizado", "asistencia", "archivo_fuente"]].copy()

assert subconjunto_validado["documento"].is_unique
assert subconjunto_validado["programa_normalizado"].eq("ADSO").all()
assert subconjunto_validado["asistencia"].lt(80).all()
assert subconjunto_validado["asistencia"].notna().all()
assert len(subconjunto_validado) == int(criterio.sum())
print(subconjunto_validado.to_string(index=False))
print("Entradas preparadas:",len(df),"| excluidas por conflicto:",int(repetido.sum()),
      "| base sin conflicto:",len(base_sin_conflictos), "| resultado:",len(subconjunto_validado))

registro_consulta = pd.DataFrame([{
    "consulta": "S4-01", "pregunta": "¿Qué registros únicos ADSO tienen asistencia <80?",
    "archivo_fuente": FUENTE.name, "umbral_didactico": 80,
    "registros_recibidos": len(df), "registros_con_clave_conflictiva": int(repetido.sum()),
    "registros_base": len(base_sin_conflictos), "registros_resultado": len(subconjunto_validado),
    "limitacion": "No corresponde a estadística oficial del SENA; datos ficticios."
}])
registro_consulta

 fila_fuente documento nombre programa_normalizado  asistencia                                archivo_fuente
           5      1004   Luis                 ADSO        70.0 seguimiento_academico_semana3_con_errores.csv
           9      1008   Juan                 ADSO        75.0 seguimiento_academico_semana3_con_errores.csv
Entradas preparadas: 19 | excluidas por conflicto: 2 | base sin conflicto: 17 | resultado: 2


,consulta,pregunta,archivo_fuente,umbral_didactico,registros_recibidos,registros_con_clave_conflictiva,registros_base,registros_resultado,limitacion
0,S4-01,¿Qué registros únicos ADSO tienen asistencia <80?,seguimiento_academico_semana3_preparado.csv,80,19,2,17,2,No corresponde a estadística oficial del SENA;...


## 8. Guardar y comprobar evidencias
Exporte resultados a `outputs/`, no sobrescriba `data/raw` ni `data/processed`. Contraste el CSV exportado con el DataFrame original, especialmente si hay tildes o identificadores.

In [9]:
salida_subconjunto = SALIDA / "semana4_adso_asistencia_baja_validado.csv"
salida_auditoria = SALIDA / "semana4_bitacora_consultas.csv"
subconjunto_validado.to_csv(salida_subconjunto,index=False,encoding="utf-8-sig")
registro_consulta.to_csv(salida_auditoria,index=False,encoding="utf-8-sig")
relectura = pd.read_csv(salida_subconjunto,dtype={"documento":"string"},encoding="utf-8-sig")
assert len(relectura)==len(subconjunto_validado)
assert relectura["documento"].tolist()==subconjunto_validado["documento"].tolist()
print("Archivos guardados:",salida_subconjunto.name,salida_auditoria.name)

Archivos guardados: semana4_adso_asistencia_baja_validado.csv semana4_bitacora_consultas.csv


## 9. Reto de transferencia - Actividad 5.7

En esta actividad se realizan tres consultas sobre el conjunto de
datos académicos ficticios utilizando isin(), between() y operadores
lógicos.

Cada consulta incluye una validación y una interpretación responsable
de los resultados.

In [10]:
# CONSULTA 1: Selección de registros por programa académico

programas_objetivo = ["ADSO", "Ciencia de datos con Python"]

consulta_1 = df.loc[
    df["programa_normalizado"].isin(programas_objetivo),
    ["documento", "nombre", "programa_normalizado"]
].copy()

# Validar que todos los programas pertenezcan a la lista
assert consulta_1["programa_normalizado"].isin(programas_objetivo).all()

print("CONSULTA 1: SELECCIÓN POR PROGRAMA")
print("Cantidad de registros:", len(consulta_1))

display(consulta_1)

CONSULTA 1: SELECCIÓN POR PROGRAMA
Cantidad de registros: 19


,documento,nombre,programa_normalizado
0,1001,Ana,ADSO
1,1002,Carlos,ADSO
2,1003,María,ADSO
3,1004,Luis,ADSO
4,1005,Sofía,ADSO
5,1006,Andrés,ADSO
6,1007,Camila,ADSO
7,1008,Juan,ADSO
8,1009,Luisa,ADSO
9,1010,Diego,ADSO


In [11]:
# CONSULTA 2: Filtrar calificaciones entre 3.0 y 4.0

consulta_2 = df.loc[
    df["nota_python"].notna()
    & df["nota_python"].between(3.0, 4.0, inclusive="both"),
    ["documento", "nombre", "nota_python"]
].copy()

# Validar que todas las notas estén dentro del rango
assert consulta_2["nota_python"].between(3.0, 4.0).all()

print("CONSULTA 2: NOTAS DE PYTHON ENTRE 3.0 Y 4.0")
print("Cantidad de registros:", len(consulta_2))

display(consulta_2)

CONSULTA 2: NOTAS DE PYTHON ENTRE 3.0 Y 4.0
Cantidad de registros: 7


,documento,nombre,nota_python
1,1002,Carlos,3.8
5,1006,Andrés,3.5
6,1007,Camila,4.0
7,1008,Juan,3.2
9,1010,Diego,3.1
15,1016,Mateo,3.6
16,1017,NaN,4.0


In [12]:
# CONSULTA 3: Registros ADSO con asistencia baja
# o calificación de Python inferior a 3.5

es_adso = df["programa_normalizado"].eq("ADSO")

asistencia_baja = (
    df["asistencia"].notna()
    & df["asistencia"].lt(80)
)

nota_baja = (
    df["nota_python"].notna()
    & df["nota_python"].lt(3.5)
)

# Combinar condiciones con &, | y ~
condicion = (
    es_adso
    & (asistencia_baja | nota_baja)
    & ~df["documento"].isna()
)

consulta_3 = df.loc[
    condicion,
    ["documento", "nombre", "programa_normalizado",
     "nota_python", "asistencia"]
].copy()

# Validar que todos los registros sean ADSO
assert consulta_3["programa_normalizado"].eq("ADSO").all()

# Validar que todos tengan documento
assert consulta_3["documento"].notna().all()

# Validar que cumplan al menos uno de los dos criterios
assert (
    (consulta_3["asistencia"].notna()
     & consulta_3["asistencia"].lt(80))
    |
    (consulta_3["nota_python"].notna()
     & consulta_3["nota_python"].lt(3.5))
).all()

print("CONSULTA 3: OPERADORES LÓGICOS")
print("Cantidad de registros:", len(consulta_3))

display(consulta_3)

CONSULTA 3: OPERADORES LÓGICOS
Cantidad de registros: 3


,documento,nombre,programa_normalizado,nota_python,asistencia
3,1004,Luis,ADSO,2.9,70.0
7,1008,Juan,ADSO,3.2,75.0
9,1010,Diego,ADSO,3.1,NaN


In [13]:
# Exportar las tres consultas del reto de transferencia

consulta_1.to_csv(
    SALIDA / "semana4_reto_consulta1_programas.csv",
    index=False,
    encoding="utf-8-sig"
)

consulta_2.to_csv(
    SALIDA / "semana4_reto_consulta2_notas.csv",
    index=False,
    encoding="utf-8-sig"
)

consulta_3.to_csv(
    SALIDA / "semana4_reto_consulta3_operadores.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Las tres consultas del reto se exportaron correctamente.")

Las tres consultas del reto se exportaron correctamente.


## 9. Reto sin solución incluida
1. Seleccione registros del programa «Ciencia de datos con Python» cuya `nota_datos` sea válida y esté entre 3.0 y 4.5. Informe filas de entrada, excluidas por conflicto y salida.
2. Formule una consulta que identifique registros con `ciudad_normalizada` faltante **o** `asistencia` faltante, sin confundir desconocido con valor cero.
3. Compare dos filtros equivalentes, uno con máscara booleana y otro con `query()`. Documente la igualdad de índices.
4. Escriba tres afirmaciones prudentes y una conclusión que **no** podría justificarse con estos datos.
5. Registre la definición del universo: registros o personas únicas, además de las limitaciones de la fuente.

In [14]:
# RETO DEL PROFESOR - ACTIVIDAD 1

# Seleccionar el programa y las notas válidas dentro del rango
mascara_reto1 = (
    df["programa_normalizado"].eq("Ciencia de datos con Python")
    & df["nota_datos"].notna()
    & df["nota_datos"].between(3.0, 4.5, inclusive="both")
)

# Excluir documentos con conflictos de identidad
resultado_reto1 = df.loc[
    mascara_reto1 & ~repetido,
    ["documento", "nombre", "programa_normalizado", "nota_datos"]
].copy()

# Validar los resultados
assert resultado_reto1["nota_datos"].between(3.0, 4.5).all()
assert resultado_reto1["programa_normalizado"].eq(
    "Ciencia de datos con Python"
).all()

print("RETO 1: FILTRADO POR PROGRAMA Y NOTA")
print("Registros de entrada:", len(df))
print("Registros excluidos por conflicto:", int(repetido.sum()))
print("Registros obtenidos:", len(resultado_reto1))

display(resultado_reto1)

RETO 1: FILTRADO POR PROGRAMA Y NOTA
Registros de entrada: 19
Registros excluidos por conflicto: 2
Registros obtenidos: 2


,documento,nombre,programa_normalizado,nota_datos
12,1013,Pedro,Ciencia de datos con Python,4.0
14,1015,Erika,Ciencia de datos con Python,3.0


In [15]:
# RETO DEL PROFESOR - ACTIVIDAD 2

# Identificar valores faltantes
ciudad_faltante = df["ciudad_normalizada"].isna()
asistencia_faltante = df["asistencia"].isna()

# Seleccionar registros con al menos un dato faltante
mascara_reto2 = ciudad_faltante | asistencia_faltante

resultado_reto2 = df.loc[
    mascara_reto2,
    [
        "documento",
        "nombre",
        "ciudad",
        "ciudad_normalizada",
        "asistencia"
    ]
].copy()

# Validar que todos los registros tengan al menos un dato faltante
assert (
    resultado_reto2["ciudad_normalizada"].isna()
    | resultado_reto2["asistencia"].isna()
).all()

print("RETO 2: CIUDAD O ASISTENCIA FALTANTE")
print("Registros encontrados:", len(resultado_reto2))

print("Registros con ciudad no normalizada:",
      int(ciudad_faltante.sum()))

print("Registros con asistencia faltante:",
      int(asistencia_faltante.sum()))

display(resultado_reto2)

RETO 2: CIUDAD O ASISTENCIA FALTANTE
Registros encontrados: 4
Registros con ciudad no normalizada: 2
Registros con asistencia faltante: 3


,documento,nombre,ciudad,ciudad_normalizada,asistencia
9,1010,Diego,Bello,Bello,NaN
10,1011,Nidia,medellín,Medellín,NaN
11,1012,Sara,BELL0,<NA>,NaN
12,1013,Pedro,NaN,<NA>,93.0


In [16]:
# RETO DEL PROFESOR - ACTIVIDAD 3

# Definir las condiciones de consulta
programa_objetivo = "ADSO"
umbral_asistencia = 80

# Consulta mediante máscara booleana
mascara = (
    df["programa_normalizado"].eq(programa_objetivo)
    & df["asistencia"].notna()
    & df["asistencia"].lt(umbral_asistencia)
)

resultado_mascara = df.loc[mascara].copy()

# Consulta equivalente utilizando query()
resultado_query = df.query(
    "programa_normalizado == @programa_objetivo "
    "and asistencia < @umbral_asistencia",
    engine="python"
).copy()

# Comparar los índices de ambos resultados
indices_iguales = resultado_mascara.index.equals(resultado_query.index)

# Validar que los resultados sean equivalentes
assert indices_iguales
assert resultado_mascara.equals(resultado_query)

print("RETO 3: COMPARACIÓN DE FILTROS")
print("Registros con máscara booleana:", len(resultado_mascara))
print("Registros con query():", len(resultado_query))
print("¿Los índices son iguales?:", indices_iguales)
print("¿Los DataFrames son iguales?:", resultado_mascara.equals(resultado_query))

display(
    resultado_mascara[
        ["documento", "nombre", "programa_normalizado", "asistencia"]
    ]
)

RETO 3: COMPARACIÓN DE FILTROS
Registros con máscara booleana: 2
Registros con query(): 2
¿Los índices son iguales?: True
¿Los DataFrames son iguales?: True


,documento,nombre,programa_normalizado,asistencia
3,1004,Luis,ADSO,70.0
7,1008,Juan,ADSO,75.0


### Actividad 4. Interpretación responsable de los resultados

**Afirmación 1:** En el conjunto de datos analizado se identificaron
dos registros del programa ADSO con asistencia inferior al 80 %:
Luis, con 70 %, y Juan, con 75 %.

**Afirmación 2:** Se identificaron cuatro registros con al menos
un problema relacionado con la ausencia de asistencia o la falta
de normalización de la ciudad.

**Afirmación 3:** Las consultas realizadas mediante una máscara
booleana y mediante query() produjeron los mismos dos registros,
lo cual se comprobó mediante la comparación de sus índices y datos.

**Conclusión que no puede justificarse:**

No podemos afirmar que todos los aprendices del programa ADSO
presentan problemas de asistencia, porque el conjunto analizado
es pequeño, contiene datos ficticios y no representa necesariamente
a toda la población estudiantil.

### Actividad 5. Universo analizado y limitaciones

**Universo de análisis:**

El archivo original contiene 20 registros académicos ficticios.
Después de retirar una fila exactamente duplicada, se obtuvieron
19 registros preparados.

Se identificaron dos registros con el mismo documento y
calificaciones diferentes. Para las consultas que requieren
identificadores únicos, estos registros se excluyeron
temporalmente, obteniendo un subconjunto de 17 registros.

Por lo tanto, los resultados deben interpretarse como
cantidades de registros y no necesariamente como cantidades
de aprendices únicos.

**Limitaciones de la fuente:**

1. Los datos son ficticios y se utilizan exclusivamente
   con fines académicos.
2. Existen documentos repetidos con información contradictoria.
3. Algunos registros presentan valores faltantes.
4. Algunas ciudades no pudieron normalizarse automáticamente.
5. Los umbrales utilizados son didácticos y no corresponden
   necesariamente a reglamentos oficiales del SENA.
6. El conjunto de datos no representa a toda la población
   estudiantil.
7. Antes de utilizar estos datos en un sistema real sería
   necesario verificar los registros conflictivos y validar
   la información con la fuente correspondiente.

# 10. Cuestionario de conocimientos - Semana 4

## Pregunta 1
**Explique la diferencia entre `df["nombre"]` y `df[["documento", "nombre"]]`.**

**Respuesta:**

La expresión `df["nombre"]` selecciona una sola columna del DataFrame y devuelve una estructura de tipo Series.

En cambio, `df[["documento", "nombre"]]` selecciona dos columnas y devuelve una estructura de tipo DataFrame.

Durante el laboratorio comprobé esta diferencia al consultar los nombres de los aprendices y al seleccionar varias columnas del archivo de seguimiento académico.

## Pregunta 2
**¿En qué se diferencian `loc` e `iloc`? Dé un ejemplo relacionado con su CSV.**

**Respuesta:**

`loc` permite seleccionar información utilizando etiquetas de filas, nombres de columnas o condiciones.

`iloc` permite seleccionar información utilizando posiciones numéricas.

En el laboratorio utilicé `loc` para consultar las columnas `documento`, `nombre` y `asistencia`. También utilicé `iloc[:4, :3]` para seleccionar las primeras cuatro filas y las primeras tres columnas del DataFrame.

Es importante recordar que las etiquetas de los índices no siempre coinciden con las posiciones de las filas.

## Pregunta 3
**¿Qué es una máscara booleana y para qué sirve en Pandas?**

**Respuesta:**

Una máscara booleana es una serie de valores True y False que permite identificar qué registros cumplen una condición.

En Pandas se utiliza para filtrar información de un DataFrame.

Durante el laboratorio utilicé una máscara booleana para identificar los registros del programa ADSO cuya asistencia era inferior al 80 %.

El resultado permitió identificar dos registros: Luis, con una asistencia del 70 %, y Juan, con una asistencia del 75 %.

## Pregunta 4
**¿Cuál es la diferencia entre `isin()` y `between()`?**

**Respuesta:**

El método `isin()` permite comprobar si los valores de una columna pertenecen a una lista de valores definidos.

Por ejemplo, lo utilicé para seleccionar registros de los programas ADSO y Ciencia de datos con Python.

Por otra parte, `between()` permite identificar valores que se encuentran dentro de un intervalo determinado.

En el laboratorio utilicé `between(3.0, 4.0)` para identificar las calificaciones de Python comprendidas entre 3.0 y 4.0, incluyendo ambos extremos.

Esta consulta encontró siete registros.

## Pregunta 5
**¿Para qué sirven los operadores lógicos `&`, `|` y `~` en Pandas?**

**Respuesta:**

Los operadores lógicos permiten combinar condiciones para realizar consultas más específicas.

- `&` significa Y: todas las condiciones deben cumplirse.
- `|` significa O: basta con que se cumpla al menos una condición.
- `~` significa NO: invierte el resultado de una condición.

Durante el laboratorio utilicé estos operadores para identificar registros del programa ADSO con asistencia inferior al 80 % o con una calificación de Python inferior a 3.5.

Esta consulta permitió identificar tres registros.

Al utilizar estos operadores en Pandas, es importante encerrar cada comparación entre paréntesis.

## Pregunta 6
**¿Qué diferencia existe entre los operadores `&` y `|` al combinar condiciones?**

**Respuesta:**

El operador `&` exige que todas las condiciones se cumplan al mismo tiempo, mientras que `|` permite seleccionar registros que cumplan al menos una de las condiciones.

Por ejemplo, durante el laboratorio utilicé `&` para identificar registros del programa ADSO con asistencia inferior al 80 %.

También utilicé `|` para encontrar registros que presentaran asistencia inferior al 80 % o una calificación de Python inferior a 3.5.

La diferencia principal es que `&` hace que la selección sea más restrictiva, mientras que `|` permite incluir registros que cumplen diferentes criterios.

## Pregunta 7
**¿Para qué sirve el método `query()` de Pandas?**

**Respuesta:**

El método `query()` permite filtrar registros de un DataFrame mediante expresiones escritas como condiciones de consulta.

Durante el laboratorio lo utilicé para identificar registros del programa ADSO con asistencia inferior al 80 %.

Después comparé los resultados obtenidos con `query()` y con una máscara booleana.

Ambos métodos devolvieron los mismos dos registros, y comprobé mediante validaciones que sus índices y datos eran iguales.

## Pregunta 8
**¿Por qué es importante validar los resultados de una consulta de datos?**

**Respuesta:**

Es importante validar los resultados porque una consulta puede ejecutarse sin errores y, aun así, producir información incorrecta si las condiciones están mal definidas.

Durante el laboratorio utilicé instrucciones `assert` para comprobar que los registros seleccionados cumplían las condiciones establecidas.

También verifiqué que los documentos no estuvieran duplicados en el subconjunto validado y que los valores de asistencia fueran conocidos e inferiores al umbral definido.

Estas validaciones ayudan a detectar errores y permiten presentar resultados más confiables.

Sin embargo, los datos utilizados son ficticios y algunos registros presentan conflictos pendientes de verificación, por lo que los resultados no deben interpretarse como estadísticas oficiales del SENA.